# Pitcher Strikeout Prediction Model
In this notebook, we will build a basic machine learning model to predict how many strikeouts a starting pitcher will get in a game. We'll use the data you already have in `mlb_db`.

### Step 1: Connect to Database and Load Data
First, we'll connect to the database and pull out every starting pitcher performance.

In [1]:
import pandas as pd
import sqlalchemy

# Connect to the database
engine = sqlalchemy.create_engine("postgresql:///mlb_db")

print("Pulling Pitching Data...")
query_pitching = """
SELECT 
    p.event_id,
    e.date,
    p.athlete_id as pitcher_id,
    p.team_id as pitcher_team_id,
    p.k as target_k
FROM event_boxscores_pitching p
JOIN events e ON p.event_id = e.event_id
WHERE p.starter = true
ORDER BY e.date ASC
"""
df_pitching = pd.read_sql(query_pitching, engine)
print(f"Loaded {len(df_pitching)} starting pitcher appearances.")
df_pitching.head()

Pulling Pitching Data...
Loaded 39089 starting pitcher appearances.


,event_id,date,pitcher_id,pitcher_team_id,target_k
0,401078885,2019-02-22 18:05:00,36061,30,1.0
1,401078885,2019-02-22 18:05:00,39869,22,3.0
2,401078698,2019-02-22 20:10:00,35096,11,2.0
3,401078698,2019-02-22 20:10:00,30465,12,2.0
4,401078892,2019-02-23 18:05:00,35026,22,4.0


### Step 2: Opponent Data (How much does the other team strike out?)
Next, we want to know how strikeout-prone the opposing team is. We can get this by summarizing the `event_boxscores_batting` table.

In [2]:
print("Pulling Team Batting Data...")
query_batting = """
SELECT 
    b.event_id,
    e.date,
    b.team_id,
    SUM(b.k) as team_strikeouts
FROM event_boxscores_batting b
JOIN events e ON b.event_id = e.event_id
GROUP BY b.event_id, e.date, b.team_id
ORDER BY e.date ASC
"""
df_batting = pd.read_sql(query_batting, engine)
df_batting.head()

Pulling Team Batting Data...


,event_id,date,team_id,team_strikeouts
0,401078885,2019-02-22 18:05:00,22,7.0
1,401078885,2019-02-22 18:05:00,30,7.0
2,401078698,2019-02-22 20:10:00,11,7.0
3,401078698,2019-02-22 20:10:00,12,7.0
4,401078890,2019-02-23 18:05:00,1,8.0


### Step 3: Matchups (Who played who?)
To connect the pitcher to the opposing team, we need to look at the `event_competitors` table.

In [3]:
query_matchups = """
SELECT event_id, team_id, home_away
FROM event_competitors
"""
df_matchups = pd.read_sql(query_matchups, engine)

# Create a dictionary mapping an event_id to the two teams playing
game_teams = df_matchups.groupby('event_id')['team_id'].apply(list).to_dict()

# Function to figure out the opposing team
def get_opp_team(row):
    teams = game_teams.get(row['event_id'], [])
    for t in teams:
        if t != row['pitcher_team_id']:
            return t
    return None

df_pitching['opp_team_id'] = df_pitching.apply(get_opp_team, axis=1)
df_pitching[['event_id', 'pitcher_id', 'pitcher_team_id', 'opp_team_id']].head()

,event_id,pitcher_id,pitcher_team_id,opp_team_id
0,401078885,36061,30,22
1,401078885,39869,22,30
2,401078698,35096,11,12
3,401078698,30465,12,11
4,401078892,35026,22,23


### Step 4: Feature Engineering (Creating our inputs)
A machine learning model can't predict today's game using today's stats (that's cheating, also known as "Lookahead Bias"). 

We need to calculate **historical averages prior to the start of the game**.

In [4]:
# Ensure strict chronological order
df_pitching = df_pitching.sort_values('date')
df_batting = df_batting.sort_values('date')

# Feature A: Pitcher's average Ks over their last 5 starts
# .shift(1) ensures today's game is excluded from the average!
df_pitching['pitcher_k_last_5'] = (
    df_pitching.groupby('pitcher_id')['target_k']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# Feature B: Opposing Team's average Ks over their last 10 games
df_batting['team_k_last_10'] = (
    df_batting.groupby('team_id')['team_strikeouts']
    .transform(lambda x: x.shift(1).rolling(window=10, min_periods=1).mean())
)

# Merge everything together so the pitcher has the opposing team's historical stats next to them
df_model = pd.merge(
    df_pitching, 
    df_batting[['event_id', 'team_id', 'team_k_last_10']], 
    left_on=['event_id', 'opp_team_id'], 
    right_on=['event_id', 'team_id'], 
    how='inner'
)

# Drop rows from early in the season before we had history established
df_model = df_model.dropna(subset=['pitcher_k_last_5', 'team_k_last_10', 'target_k'])

print(f"Final Model Dataset Ready! Total Matchups: {len(df_model)}")
df_model[['pitcher_id', 'opp_team_id', 'pitcher_k_last_5', 'team_k_last_10', 'target_k']].head()

Final Model Dataset Ready! Total Matchups: 37478


,pitcher_id,opp_team_id,pitcher_k_last_5,team_k_last_10,target_k
155,30465,5,2.0,7.25,3.0
158,34955,12,1.0,7.20,2.0
159,35096,19,2.0,4.75,0.0
161,31593,4,1.0,7.40,2.0
172,33870,10,2.0,6.75,1.0


### Step 5: Train the Algorithm
We split the data into 80% Training Data (The Past) and 20% Testing Data (The Future). We'll use a `RandomForestRegressor`, which handles sports statistics very well out-of-the-box.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. Define inputs (X) and what we are predicting (y)
features = ['pitcher_k_last_5', 'team_k_last_10']
X = df_model[features]
y = df_model['target_k']

# 2. Split Data chronologically
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. Train the Model
model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

print("Model Trained Successfully!")

### Step 6: Evaluation & Predictions
Let's see how well it learned. We'll ask it to predict the 20% of data it has never seen, and calculate the error.

In [ ]:
# Make predictions
predictions = model.predict(X_test)

# Calculate Average Error
mae = mean_absolute_error(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.2f} Strikeouts")
print(f"This means on average, our model's guess is off by about {mae:.2f} Ks.\n")

print("What the model cares about most (Feature Importances):")
for feature, importance in zip(features, model.feature_importances_):
    print(f"- {feature}: {importance:.2%}")

print("\nLet's look at 5 random predictions from the test set:")
examples = X_test.sample(5, random_state=123).copy()
examples['Actual Ks Achieved'] = y_test.loc[examples.index]
examples['Predicted Ks'] = model.predict(examples[features]).round(1)
examples[['pitcher_k_last_5', 'team_k_last_10', 'Predicted Ks', 'Actual Ks Achieved']]